### Case Study 1: Using Regex, Tokenizing, POS tagging, and Knowledge-based (standard UMLS concepts) for Unstructured Clinical Notes
In this case study, we will walk through how to analyze unstructured clinical notes. The data that we will be using are a sample of 20 discharge summary notes and 20 radiology reports found in Kaggle's [Clinical Documents on Syndromes Disease](https://www.kaggle.com/datasets/muhammadimran112233/clinical-documents-on-syndromes-disease/data), and a sample history of present illness (HPI) in plain text. 

The problem we are trying to solve is finding specific segments of text in clinical documents, and map plain text HPI to standard medical concepts in UMLS. The approach is to use regex so that we preserve privacy, maintain predictability, and remain efficient. For the HPI data, we will build a NLP pipeline with knowledge-based lookup.

LLM Statement: Used LLM to understand regex, define crtieria carefully, trace code to verify it answers all of the problem statements, and understand the limitations of the proposed solution.

In [1]:
import os
import re # regular expressions
import csv

import nltk

# sent_tokenize and word_tokenize: punkt / punkt_tab
# pos_tag: averaged_perceptron_tagger or averaged_perceptron_tagger_eng
# nltk version
for resource in ["punkt", "averaged_perceptron_tagger"]:
    nltk.download(resource, quiet=True)
import requests # call and load for UMLS api key safely
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.environ["UMLS_API_KEY"]

In [2]:
DATA_DIR = "dataset"

def write_tsv(path, header, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(header)
        writer.writerows(rows)

#### Part 1: Discharge Summaries

Find the discharge *diagnoses* if they exist in the document, and output each of the diagnoses into a file as the following:

For each diagnosis identified in each document, enter each of them into the tab delimited file as a row with 4 columns (Document Name, start pos, end pos, diagnosis text).

For this, we will be using the ``find_dcdx_section(fname)`` and ``find_dcdx_list(input_text)`` to find the section and find the diagnoses, respectively.

In [3]:
# possible headers
# DISCHARGE DIAGNOSIS/DIAGNOSES or DIAGNOSIS/DIAGNOSES ON DISCHARGE
# Nothing else on the line except spaces counts as a header: [ \t]*$
# optional colon: \s*:?
DISCHARGE_DIAGNOSES_HEADER = re.compile(r"^(?:DISCHARGE\s+DIAGNOS[EI]S|DIAGNOSES\s+ON\s+DISCHARGE)\s*:?[ \t]*$", re.M)

# Discharge notes either are number are not
# With space after . or not
# "1.  Syncope."  or  "2 . End-stage COPD." ~ 3979
# line starting with numbers + period + space + diagnosis + optional period + spaces + end of line
NUMBERED_ITEM = re.compile(r"^[ \t]*\d+[ \t]*\.[ \t]*(?P<dx>\S.*?)\.?[ \t]*$", re.M)


# No numbering 
# first blank line
# some indent + diangoses + optional period + spaces + end of line
FIRST_LINE = re.compile(r"^[ \t]*(?P<dx>\S.*?)\.?[ \t]*$", re.M)


# Any other all-caps header line ends the section
# capital letter only with some symbols + optional colon
NEXT_HEADER = re.compile(r"^[A-Z][A-Z /&,()-]+:?[ \t]*$", re.M)

def find_dcdx_section(fname):
    """
    Takes a file name as a parameter
    Return the discharge diagnoses sections as a list of section texts
    """
    with open(fname, encoding="utf-8") as f:
        text = f.read()
    sections = []
    for header in DISCHARGE_DIAGNOSES_HEADER.finditer(text):
        nxt = NEXT_HEADER.search(text, header.end())
        end = nxt.start() if nxt else len(text)
        sections.append(text[header.end():end])
    return sections


def find_dcdx_list(input_text):
    """
    Takes in the input text     
    Return (start, end, diagnosis) tuples
    e.g.
    first diagnosis of 3880 becomes 6-61 (relative to discharge diagnoses section) instead of 163-218 (whole file)
    """
    matches = list(NUMBERED_ITEM.finditer(input_text))
    if not matches:
        first = FIRST_LINE.search(input_text)
        matches = [first] if first else []
    return [(m.start("dx"), m.end("dx"), m.group("dx")) for m in matches]

In [4]:
discharge_summary_files = sorted(f for f in os.listdir(DATA_DIR) if re.match(r"^\d+_Discharge Summary\.txt$", f))

part1_rows = []
for fname in discharge_summary_files:
    for section in find_dcdx_section(os.path.join(DATA_DIR, fname)):
        for start, end, dx in find_dcdx_list(section):
            part1_rows.append((fname, start, end, dx))

write_tsv("part1_discharge_diagnoses.tsv", ["Document Name", "Start", "End", "Diagnosis"], part1_rows)

In [5]:
import pandas as pd
df = pd.read_csv("part1_discharge_diagnoses.tsv", sep="\t", keep_default_na=False)
df.head()

,Document Name,Start,End,Diagnosis
0,3880_Discharge Summary.txt,6,61,Severe menometrorrhagia unresponsive to medica...
1,3880_Discharge Summary.txt,68,81,Severe anemia
2,3880_Discharge Summary.txt,88,114,Symptomatic fibroid uterus
3,3880_Discharge Summary.txt,121,165,Extensive adenomyosis by pathological report
4,3881_Discharge Summary.txt,3,57,Enlarged prostate with benign bladder neck obs...


#### Part 2: Radiology Reports

Find the `IMPRESSION:` section, skip reports without one, and POS-tag it with NLTK to pull out NN/NNS/NNP/NNPS tokens.

For context
- NN: Noun
- NNS: Noun, plural
- NNP: Proper noun, singular
- NNPS: Proper noun, plural

In [6]:
# keeping this for report 12, the brackets shouldn't be counted as nouns
# sometimes we get MIMIC de-identification placeholders [**2145-2-27**]
DEID_TAG = re.compile(r"\[\*\*.*?\*\*\]")  
NOUN_TAGS = {"NN", "NNS", "NNP", "NNPS"}


def find_imp_section(fname):
    """
    Takes in the filename
    Return the IMPRESSION section (if it has one)
    """
    with open(fname, encoding="utf-8") as f:
        text = f.read()
    IMP_SECTION_RE = re.compile(r"^[ \t]*IMPRESSION[ \t]*:\s*(?P<imp>.+?)(?=\n[ \t]*\n|\Z)", re.M | re.S)
    impression_content = IMP_SECTION_RE.search(text)
    if not impression_content:
        return None
    imp = DEID_TAG.sub(" ", impression_content.group("imp"))
    return re.sub(r"\s+", " ", imp).strip()


def find_noun_list(input_text):
    """
    Takes the input text
    Return a list of (tag, token) pairs for every noun in the text.
    """
    tagged = nltk.pos_tag(nltk.word_tokenize(input_text))
    return [(tag, word) for word, tag in tagged if tag in NOUN_TAGS]

In [7]:
radiology_files = sorted(
    (f for f in os.listdir(DATA_DIR) if re.match(r"^radiology-report\d+\.txt$", f)),
    key=lambda f: int(re.search(r"\d+", f).group()),
)

part2_rows = []
for fname in radiology_files:
    impression_section = find_imp_section(os.path.join(DATA_DIR, fname))
    if impression_section is None:
        continue
    for tag, word in find_noun_list(impression_section):
        part2_rows.append((fname, tag, word))

write_tsv("part2_impression_nouns.tsv", ["Document Name", "POS Tag", "Text"], part2_rows)

In [8]:
df2 = pd.read_csv("part2_impression_nouns.tsv", sep="\t", keep_default_na=False)
df2.head()

,Document Name,POS Tag,Text
0,radiology-report3.txt,NNS,Findings
1,radiology-report3.txt,NN,recurrence
2,radiology-report3.txt,NN,increase
3,radiology-report3.txt,NN,number
4,radiology-report3.txt,NN,celiac


#### Part 3: HPI + UMLS

Key concepts:
- **Sentence and word tokenization**: split the HPI into sentences, then into tokens (`nltk.sent_tokenize`, `nltk.word_tokenize`)
- **Part-of-speech (POS) tagging**: label each token with a Penn Treebank tag and keep the nouns (NN/NNS) as candidate medical terms
- **Concept normalization / entity linking**: map free-text words to standard concepts (CUIs) in UMLS, restricted to the SNOMED CT vocabulary
- **Querying a terminology API**: call the UMLS UTS REST API with exact-match search, cache repeated lookups
- **Limitations of a bag-of-nouns approach**: multi-word terms get split up and negation/context is lost

Here we will switch over to study history of present illness (HPI). For this data, we will do two things.

**3a.** Sentence-tokenize the HPI, POS-tag each sentence, keep NN/NNS tokens.

**3b.** Look up each noun in the UMLS UTS search API (`searchType=exact`, `sabs=SNOMEDCT_US`) and write the non-empty results to a TSV. UMLS is useful to connect the HPI words to standard medical concepts.

In [9]:
with open("hpi-example.txt", encoding="utf-8") as f:
    hpi_text = f.read()

hpi_nouns = []
for sentence in nltk.sent_tokenize(hpi_text):
    for word, tag in nltk.pos_tag(nltk.word_tokenize(sentence)):
        if tag in {"NN", "NNS"}:
            hpi_nouns.append((word, tag))

pd.DataFrame(hpi_nouns, columns=["Text", "POS Tag"])

,Text,POS Tag
0,History,NN
1,admission,NN
2,year,NN
3,woman,NN
4,state,NN
...,...,...
91,family,NN
92,history,NN
93,premature,NN
94,cholesterol,NN


In [ ]:
UTS_SEARCH_URL = "https://uts-ws.nlm.nih.gov/rest/search/current"


def umls_exact_search(term):
    params = {"string": term, "searchType": "exact", "sabs": "SNOMEDCT_US", "apiKey": API_KEY}
    resp = requests.get(UTS_SEARCH_URL, params=params, timeout=30)
    resp.raise_for_status()
    # Older API versions return a single {"ui": "NONE"} placeholder instead of an empty list
    return [r for r in resp.json()["result"]["results"] if r["ui"] != "NONE"]


cache = {}
part3_rows = []
for word, _ in hpi_nouns:  # repeated nouns get one row each
    if word not in cache:
        cache[word] = umls_exact_search(word)
    for hit in cache[word]:
        part3_rows.append((word, hit["ui"], hit["name"], hit["rootSource"]))

write_tsv("part3_umls_concepts.tsv", ["INPUT", "CUI", "NAME", "SOURCE VOCAB"], part3_rows)

In [11]:
df3 = pd.read_csv("part3_umls_concepts.tsv", sep="\t", keep_default_na=False)
df3.head()

,INPUT,CUI,NAME,SOURCE VOCAB
0,History,C0262926,Medical History,MTH
1,year,C0439234,year,MTH
2,woman,C0043210,Woman,MTH
3,state,C1301808,Geographic state,MTH
4,health,C0018684,Health,MTH


#### Limitations of the current NLP pipeline and knowledge-based lookup
What we'll find is the weakness of looking up single nouns. For example, "chest pain" should really be one concept instead of two separate words. Also negations are very important, so observations like no sweating or nausea, etc. shouldn't show up without the negation context. One one to mitigate these limitations is to use handle these cases by building candidate n-grams and flag negation within the sentence.